# C04 — Inferência Local ou Remota

Comparação entre inferência remota via API (Anthropic) e inferência local com Ollama / HuggingFace.

## 1. Inferência remota — API Anthropic

In [ ]:
import os, time
import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

PROMPT = "Explique o conceito de overfitting em machine learning em 3 frases."

inicio = time.time()
resp = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=256,
    messages=[{"role": "user", "content": PROMPT}]
)
latencia = time.time() - inicio

print(f"[Remota — Claude Haiku] Latência: {latencia:.2f}s")
print(resp.content[0].text)

## 2. Inferência local — Ollama

> Pré-requisito: `ollama serve` rodando localmente e modelo baixado com `ollama pull llama3.2`

In [ ]:
%pip install ollama -q

In [ ]:
import ollama

try:
    inicio = time.time()
    resp_local = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": PROMPT}]
    )
    latencia_local = time.time() - inicio
    print(f"[Local — Llama 3.2] Latência: {latencia_local:.2f}s")
    print(resp_local["message"]["content"])
except Exception as e:
    print(f"Ollama não disponível: {e}")
    print("Inicie com: ollama serve && ollama pull llama3.2")

## 3. Inferência local — HuggingFace Transformers (CPU)

In [ ]:
%pip install transformers torch -q

In [ ]:
from transformers import pipeline

# Modelo leve para demonstração em CPU
generator = pipeline("text-generation", model="distilgpt2", max_new_tokens=80)

inicio = time.time()
saida = generator("Overfitting in machine learning is")
latencia_hf = time.time() - inicio

print(f"[Local — distilgpt2] Latência: {latencia_hf:.2f}s")
print(saida[0]["generated_text"])

## 4. Comparativo

In [ ]:
import pandas as pd

dados = [
    {"Abordagem": "Remota (Claude Haiku)", "Latência (s)": round(latencia, 2), "Custo": "Por token",  "Hardware": "Nenhum"},
    {"Abordagem": "Local (Ollama/Llama)",  "Latência (s)": "—",               "Custo": "Gratuito",  "Hardware": "GPU/CPU"},
    {"Abordagem": "Local (HuggingFace)",   "Latência (s)": round(latencia_hf, 2), "Custo": "Gratuito", "Hardware": "CPU"},
]

pd.DataFrame(dados)